# Argus — YOLO nano baseline (Ultralytics)

The off-the-shelf reference for our from-scratch detector in `12_tiny_grid_detector.ipynb`: a
**pretrained YOLO nano** (COCO weights), fine-tuned on the same face-crop dataset, splits and
classes, and scored by the same evaluation code. If our tiny CNN gets close to this at a
fraction of the size and latency, that's the result.

Two models:
- **`yolo11n`**: the current Ultralytics nano, about 2.6 M parameters. Primary.
- **`yolov5nu`**: the YOLOv5-nano lineage (anchor-free "u" head), about 2.6 M parameters.

**Input.** The same `IMG = 160` face crops. Ultralytics reads the 1-channel JPEGs as 3-channel
BGR (gray replicated), so the network sees gray. HSV augmentation is turned off, since there is
no colour to jitter.

**Licence note.** Ultralytics is **AGPL-3.0**. That's fine for this academic project, but it
matters before any commercial deployment.

**Reads:** `dataset/parts_det.zip`.

**Writes:**
- `models/parts_det/<model>_*` (best `.pt` + int8 `.tflite`);
- `<name>_metrics.json` for notebook 14's comparison table.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3,<9"], check=False)

## Setup

**Getting the dataset in (Colab).** Upload `parts_det.zip`, built by
`src/dataset/scripts/build_detection_dataset.py`, **straight into the session**. Either drag it
into the Files panel (left sidebar → 📁 → upload, so it lands at `/content/parts_det.zip`), or
run the Loading cell and use the upload button it shows if the zip isn't found. Drive isn't
needed for the dataset.

**Keeping the results.** A Colab session's disk is wiped when it ends.
- With `USE_DRIVE = True` (the default), Drive is mounted only to save trained models and
  metrics to `MyDrive/Argus/models/parts_det/`. They survive the session, and notebook 14's
  comparison table can read all three notebooks' results.
- With `USE_DRIVE = False`, everything is saved under `/content/models/parts_det/` and the last
  cell zips it for download.

**Locally:** set `ARGUS_DET_ZIP`, and optionally `ARGUS_PROJECT_FOLDER` / `ARGUS_WORK_DIR`.
`ARGUS_SMOKE=1` turns every training run into a few-step smoke test on a small subset. It's used
to run the notebook end to end on a CPU before spending GPU time.

**This cell is shared verbatim by notebooks 12 / 13 / 14** so their numbers are comparable;
change it in all three or none.

In [ ]:
import os
import sys
import json
import time
import shutil
import zipfile
import datetime

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("ARGUS_SMOKE", "0") == "1"
USE_DRIVE = True          # Colab only: save models/metrics to Drive so they outlive the session

if IN_COLAB:
    WORK_DIR = "/content"
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        project_folder = "/content/drive/MyDrive/Argus"
    else:
        project_folder = "/content"
else:
    project_folder = os.environ.get("ARGUS_PROJECT_FOLDER", os.path.expanduser("~/argus-colab-mirror"))
    WORK_DIR = os.environ.get("ARGUS_WORK_DIR", "/tmp/argus_work")

models_folder = f"{project_folder}/models/parts_det"
# Where the dataset zip may be, in order: explicit env var, uploaded into the session, on Drive.
DET_ZIP_CANDIDATES = [p for p in (os.environ.get("ARGUS_DET_ZIP"), f"{WORK_DIR}/parts_det.zip",
                                  f"{project_folder}/dataset/parts_det.zip") if p]
os.makedirs(models_folder, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
VERSION_STR = datetime.datetime.now().strftime("%Y%m%d_%H%M")

# Dataset contract (src/dataset/argus_dataset/config.py: DET_CLASSES, DET_IMG_SIZE = 192 stored).
CLASSES = ["open_eye", "closed_eye", "yawn", "head_straight", "head_down"]
NUM_CLASSES = len(CLASSES)
EYE_CLASSES, YAWN_CLASS, HEAD_CLASSES = (0, 1), 2, (3, 4)

IMG = 160                 # model input side (stored crops are 192 px, resized here)
VARIANT = "mix"           # "gray" | "ir" | "mix" (train: random gray/IR blend per image)
EVAL_VARIANT = "ir"       # what val/test images look like: the cabin camera is a Pi NoIR
INCLUDE_CLOSEUPS = True   # MRL + Roboflow eye/mouth close-ups (whole image = one box)
INCLUDE_PSEUDO_EYES = True  # images whose skipped eyes were pseudo-labelled from landmarks
CONF_THR = 0.25           # operating point for P/R, eye-state and head accuracy
SEED = 42
np.random.seed(SEED)
print(f"IN_COLAB={IN_COLAB} SMOKE={SMOKE} USE_DRIVE={USE_DRIVE}\nmodels -> {models_folder}")

## Loading the detection dataset

The zip is found in this order:
1. `ARGUS_DET_ZIP`;
2. `/content/parts_det.zip`, i.e. uploaded into the session;
3. `MyDrive/Argus/dataset/parts_det.zip`;
4. otherwise an upload button.

It's unzipped to local session disk once. Training reads the local files, never Drive's FUSE
mount: thousands of small reads off Drive time out, as `07`/`11` found.

The zip is YOLO format:
- `gray/` and `ir/`, each with `{train,valid,test}/{images,labels}`;
- `index.csv`, one row per source image with `status`, `kind`, `source`, `group_id` and `split`.

**Splits are by group:** an original image plus its augmented copies and near-duplicates, or one
MRL subject. They were assigned when the dataset was built. Never re-split by image here.

Each label line is `class cx cy w h`, normalised to 0–1.

**This cell is shared verbatim by notebooks 12 / 13 / 14.**

In [ ]:
LOCAL_DIR = f"{WORK_DIR}/parts_det"


def find_dataset_zip():
    for p in DET_ZIP_CANDIDATES:
        if os.path.isfile(p) and zipfile.is_zipfile(p):
            return p
    if IN_COLAB:   # not uploaded yet: show an upload button (Files panel drag-and-drop is faster for 371 MB)
        from google.colab import files
        print("parts_det.zip not found — choose it in the upload button below.")
        for name in files.upload():
            if name.endswith(".zip"):
                dest = f"{WORK_DIR}/parts_det.zip"
                if os.path.abspath(name) != dest:
                    shutil.move(name, dest)
                return dest
    raise FileNotFoundError(f"parts_det.zip not found in {DET_ZIP_CANDIDATES}")


if not os.path.isfile(f"{LOCAL_DIR}/index.csv"):
    DET_ZIP = find_dataset_zip()
    print(f"Extracting {DET_ZIP} -> {LOCAL_DIR}")
    with zipfile.ZipFile(DET_ZIP) as zf:
        zf.extractall(LOCAL_DIR)

index = pd.read_csv(f"{LOCAL_DIR}/index.csv", dtype=str, keep_default_na=False)
ok = index[index.status == "ok"].copy()
if not INCLUDE_CLOSEUPS:
    ok = ok[ok.kind != "closeup"]
if not INCLUDE_PSEUDO_EYES:
    ok = ok[ok.pseudo_eyes.isin(["", "0"])]


def read_labels(label_rel):
    """YOLO label file -> (classes int32 (n,), boxes float32 (n,4) as cx,cy,w,h)."""
    rows = [l.split() for l in open(f"{LOCAL_DIR}/{label_rel}").read().splitlines() if l.strip()]
    if not rows:
        return np.zeros(0, np.int32), np.zeros((0, 4), np.float32)
    a = np.array(rows, np.float32)
    return a[:, 0].astype(np.int32), a[:, 1:5]


splits = {}
for s in ("train", "valid", "test"):
    df = ok[ok.split == s].reset_index(drop=True)
    if SMOKE:   # a few images of every (source, kind) so every code path still runs
        df = (df.groupby(["source", "kind"], group_keys=False)
                .apply(lambda g: g.sample(min(len(g), 24), random_state=SEED)).reset_index(drop=True))
    labels = [read_labels(p) for p in df.label_path]
    df["classes"] = [c for c, _ in labels]
    df["boxes"] = [b for _, b in labels]
    splits[s] = df

summary = pd.DataFrame({
    s: {"images": len(d), **{c: int(sum(int((a == i).sum()) for a in d.classes))
                             for i, c in enumerate(CLASSES)}}
    for s, d in splits.items()})
print(summary)


def image_path(row, variant):
    return f"{LOCAL_DIR}/{row['gray_path'] if variant == 'gray' else row['ir_path']}"


def gts_for(df):
    """Ground truth per image as xyxy boxes, for the shared evaluation."""
    out = {}
    for r in df.itertuples():
        b = r.boxes
        out[r.image_id] = (np.c_[b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2,
                                 b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], r.classes)
    return out

## Shared evaluation

Each model's output is converted to a common form: per image, `xyxy` boxes (0–1), scores and
class ids, after NMS. It is then scored here, so 12, 13 and 14 report **identically defined**
numbers.

- **mAP@0.5 and mAP@0.5:0.95** use COCO-style 101-point interpolation, per class.
- **P / R per class** are taken at `CONF_THR` and IoU ≥ 0.5.
- **Eye-state accuracy** is what the temporal stage will actually consume. Each ground-truth eye
  is matched to the best-IoU predicted eye box of either eye class (IoU ≥ 0.5, score ≥
  `CONF_THR`). From that we get:
  - `eye_recall`: the share of eyes found;
  - `eye_state_acc`: the open/closed correctness among found eyes;
  - `eye_state_overall`: correct over all eyes.
- **Head accuracy** is defined the same way over `head_straight` / `head_down`.

`report()` also splits these by `kind` (face / closeup) and `source` (roboflow / mrl_eye),
because MRL close-ups are easy and must not hide weak face-crop results.

**Shared verbatim by notebooks 12 / 13 / 14.**

In [ ]:
def iou_xyxy(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    x0 = np.maximum(a[:, None, 0], b[None, :, 0]); y0 = np.maximum(a[:, None, 1], b[None, :, 1])
    x1 = np.minimum(a[:, None, 2], b[None, :, 2]); y1 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)
    area = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)


def nms_per_class(boxes, scores, classes, iou_thr=0.5, max_det=30):
    keep = []
    for c in np.unique(classes):
        idx = np.where(classes == c)[0]
        idx = idx[np.argsort(-scores[idx])]
        while len(idx):
            i = idx[0]
            keep.append(i)
            if len(idx) == 1:
                break
            idx = idx[1:][iou_xyxy(boxes[i:i + 1], boxes[idx[1:]])[0] < iou_thr]
    keep = np.array(sorted(keep, key=lambda i: -scores[i])[:max_det], np.int64)
    return boxes[keep], scores[keep], classes[keep]


def _ap(recall, precision):
    r = np.concatenate([[0.0], recall, [1.0]]); p = np.concatenate([[1.0], precision, [0.0]])
    p = np.flip(np.maximum.accumulate(np.flip(p)))
    return float(np.mean(np.interp(np.linspace(0, 1, 101), r, p)))


def average_precision(preds, gts, cls, iou_thrs):
    """AP of one class at each IoU threshold over a set of images."""
    items, n_gt = [], 0
    for iid, (gb, gc) in gts.items():
        g = gb[gc == cls]; n_gt += len(g)
        pb, ps, pc = preds.get(iid, (np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)))
        m = pc == cls
        items.append((g, pb[m], ps[m]))
    if n_gt == 0:
        return [float("nan")] * len(iou_thrs)
    out = []
    for t in iou_thrs:
        scores, tps = [], []
        for g, pb, ps in items:
            order = np.argsort(-ps); pb, ps = pb[order], ps[order]
            ious = iou_xyxy(pb, g); used = np.zeros(len(g), bool)
            for k in range(len(pb)):
                j = int(np.argmax(ious[k])) if len(g) else -1
                hit = j >= 0 and ious[k, j] >= t and not used[j]
                if hit:
                    used[j] = True
                scores.append(ps[k]); tps.append(hit)
        if not scores:
            out.append(0.0); continue
        order = np.argsort(-np.array(scores)); tp = np.array(tps)[order]
        ctp, cfp = np.cumsum(tp), np.cumsum(~tp)
        out.append(_ap(ctp / n_gt, ctp / np.maximum(ctp + cfp, 1)))
    return out


def state_accuracy(preds, gts, group, conf=CONF_THR, iou_thr=0.5):
    """For GT boxes whose class is in `group`: found-rate and state accuracy (see markdown)."""
    total = found = correct = 0
    for iid, (gb, gc) in gts.items():
        m = np.isin(gc, group)
        if not m.any():
            continue
        pb, ps, pc = preds.get(iid, (np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)))
        k = np.isin(pc, group) & (ps >= conf)
        ious = iou_xyxy(gb[m], pb[k])
        for g_i, cls in enumerate(gc[m]):
            total += 1
            if ious.shape[1] and ious[g_i].max() >= iou_thr:
                found += 1
                correct += int(pc[k][int(np.argmax(ious[g_i]))] == cls)
    return {"n": total, "recall": found / max(total, 1), "state_acc": correct / max(found, 1),
            "state_overall": correct / max(total, 1)}


def pr_at_conf(preds, gts, cls, conf=CONF_THR, iou_thr=0.5):
    tp = fp = n_gt = 0
    for iid, (gb, gc) in gts.items():
        g = gb[gc == cls]; n_gt += len(g)
        pb, ps, pc = preds.get(iid, (np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)))
        p = pb[(pc == cls) & (ps >= conf)]
        ious = iou_xyxy(p, g); used = np.zeros(len(g), bool)
        for k in range(len(p)):
            j = int(np.argmax(ious[k])) if len(g) else -1
            if j >= 0 and ious[k, j] >= iou_thr and not used[j]:
                used[j] = True; tp += 1
            else:
                fp += 1
    return tp / max(tp + fp, 1), tp / max(n_gt, 1)


def evaluate(preds, gts):
    thrs = np.arange(0.5, 0.96, 0.05)
    aps = {c: average_precision(preds, gts, i, thrs) for i, c in enumerate(CLASSES)}
    res = {
        "map50": float(np.nanmean([a[0] for a in aps.values()])),
        "map50_95": float(np.nanmean([np.mean(a) for a in aps.values()])),
        "ap50": {c: a[0] for c, a in aps.items()},
        "pr_at_conf": {c: dict(zip(("precision", "recall"), pr_at_conf(preds, gts, i)))
                       for i, c in enumerate(CLASSES)},
        "eye": state_accuracy(preds, gts, EYE_CLASSES),
        "head": state_accuracy(preds, gts, HEAD_CLASSES),
    }
    return res


def report(preds, df, title=""):
    """Overall + per-kind / per-source evaluation of `preds` on the images of `df`."""
    out = {"all": evaluate(preds, gts_for(df))}
    for col in ("kind", "source"):
        for v, sub in df.groupby(col):
            out[f"{col}={v}"] = evaluate(preds, gts_for(sub))
    rows = {k: {"mAP50": r["map50"], "mAP50-95": r["map50_95"],
                **{f"AP50 {c}": r["ap50"][c] for c in CLASSES},
                "eye recall": r["eye"]["recall"], "eye state acc": r["eye"]["state_acc"],
                "head recall": r["head"]["recall"], "head acc": r["head"]["state_acc"]}
            for k, r in out.items()}
    print(f"== {title}")
    print(pd.DataFrame(rows).T.round(3).to_string())
    return out

## Shared latency benchmark and metrics file

The Pi runs **int8 TFLite**. `bench_tflite` times one image through a `.tflite` model with
`num_threads` set to 1, 2 and 4: the Pi 5 has 4 cores, and small models often don't speed up
with more threads, which is the MobileNet-on-Pi question. Colab's CPU is only a *proxy*. The
saved `.tflite` is what gets timed on the real Pi with `benchmark_model`.

`save_metrics` writes `models/parts_det/<name>_metrics.json`. Notebook 14's last cell reads
every such file into one comparison table.

**Shared verbatim by notebooks 12 / 13 / 14.**

In [ ]:
import tensorflow as tf


def bench_tflite(path, threads=(1, 2, 4), runs=200, warmup=20):
    out = {}
    for n in threads:
        it = tf.lite.Interpreter(model_path=str(path), num_threads=n)
        it.allocate_tensors()
        inp = it.get_input_details()[0]
        x = np.random.rand(*inp["shape"]).astype(np.float32)
        if inp["dtype"] != np.float32:   # integer-input model: feed in its quantised domain
            s, z = inp["quantization"]
            x = np.clip(np.round(x / s + z), np.iinfo(inp["dtype"]).min, np.iinfo(inp["dtype"]).max).astype(inp["dtype"])
        ts = []
        for k in range(warmup + (5 if SMOKE else runs)):
            t0 = time.perf_counter()
            it.set_tensor(inp["index"], x); it.invoke()
            if k >= warmup:
                ts.append((time.perf_counter() - t0) * 1000)
        out[str(n)] = round(float(np.median(ts)), 3)
    print(f"latency (median ms) {os.path.basename(str(path))}: {out}")
    return out


def _jsonable(o):
    if isinstance(o, dict):
        return {k: _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_jsonable(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if np.isnan(o) else float(o)
    if isinstance(o, np.integer):
        return int(o)
    return o


def save_metrics(name, notebook, **fields):
    rec = {"name": name, "notebook": notebook, "variant": VARIANT, "eval_variant": EVAL_VARIANT,
           "img": IMG, "smoke": SMOKE, "version": VERSION_STR, **fields}
    path = f"{models_folder}/{name}_metrics.json"
    with open(path, "w") as f:
        json.dump(_jsonable(rec), f, indent=2)
    print(f"-> {path}")
    return path

## Dataset files for Ultralytics

Ultralytics accepts a `.txt` list of image paths per split and finds each label by swapping
`/images/` for `/labels/`. Writing the lists from `splits` means its training sees exactly the
images the Setup filters selected (close-ups, pseudo-eyes, smoke subset), the same as 12/14.

Training images follow `VARIANT`:
- `gray` or `ir` selects that folder;
- `mix` lists **both** versions of every training image. That's the closest file-based
  equivalent of 12/14's random per-image blend, and Ultralytics can't blend.

Validation and test use `EVAL_VARIANT`.

In [ ]:
def write_list(df, variant, name):
    path = f"{WORK_DIR}/yolo_{name}.txt"
    rows = [image_path(r, variant) for _, r in df.iterrows()]
    with open(path, "w") as f:
        f.write("\n".join(rows) + "\n")
    return path


train_variants = ["gray", "ir"] if VARIANT == "mix" else [VARIANT]
train_lists = [write_list(splits["train"], v, f"train_{v}") for v in train_variants]
YOLO_YAML = f"{WORK_DIR}/parts_det_ultralytics.yaml"
with open(YOLO_YAML, "w") as f:
    f.write(f"train: {json.dumps(train_lists)}\n"
            f"val: {write_list(splits['valid'], EVAL_VARIANT, 'valid')}\n"
            f"test: {write_list(splits['test'], EVAL_VARIANT, 'test')}\n"
            f"nc: {NUM_CLASSES}\nnames: {json.dumps(CLASSES)}\n")
print(open(YOLO_YAML).read())

## Training and evaluation

Fine-tuning settings:
- `imgsz=IMG` and default mosaic;
- `fliplr=0.5`, because the classes have no left/right;
- `hsv_s=hsv_h=0`, because the images are gray;
- early stopping with patience 20 on Ultralytics' own val fitness.

For scoring, predictions are taken at `conf=0.001` with NMS IoU 0.5, matching what 12/14 hand to
the shared evaluation. They are converted to the common `{image_id: (xyxy, scores, classes)}`
form, so mAP, eye-state and head numbers are defined exactly as in the other notebooks.
Ultralytics' own mAP is printed too, for reference only.

In [ ]:
from ultralytics import YOLO


def ultra_predictions(model, df, variant=None):
    preds = {}
    paths_ = [image_path(r, variant or EVAL_VARIANT) for _, r in df.iterrows()]
    for k in range(0, len(paths_), 64):
        for iid, res in zip(df.image_id[k:k + 64],
                            model.predict(paths_[k:k + 64], imgsz=IMG, conf=0.001, iou=0.5,
                                          max_det=30, verbose=False)):
            b = res.boxes
            preds[iid] = (b.xyxyn.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy().astype(int))
    return preds


def run_ultralytics(weights, epochs=100):
    name = weights.replace(".pt", "")
    model = YOLO(weights)
    model.train(data=YOLO_YAML, imgsz=IMG, epochs=1 if SMOKE else epochs, batch=64,
                patience=20, fliplr=0.5, hsv_h=0.0, hsv_s=0.0, seed=SEED, deterministic=True,
                project=f"{WORK_DIR}/ultralytics", name=name, exist_ok=True,
                workers=2 if SMOKE else 8, plots=False, verbose=False)
    best = YOLO(f"{WORK_DIR}/ultralytics/{name}/weights/best.pt")
    val = report(ultra_predictions(best, splits["valid"]), splits["valid"], f"{name} / valid")
    test_float = report(ultra_predictions(best, splits["test"]), splits["test"], f"{name} / TEST float")

    pt_path = f"{models_folder}/{name}_{VERSION_STR}.pt"
    shutil.copy(f"{WORK_DIR}/ultralytics/{name}/weights/best.pt", pt_path)
    tflite_path, test_int8, latency = None, None, None
    try:  # int8 TFLite export pulls in onnx/onnx2tf; keep the notebook going if it fails
        out = best.export(format="tflite", int8=True, imgsz=IMG, data=YOLO_YAML, verbose=False)
        tflite_path = f"{models_folder}/{name}_{VERSION_STR}_int8.tflite"
        shutil.copy(out, tflite_path)
        test_int8 = report(ultra_predictions(YOLO(out, task="detect"), splits["test"]),
                           splits["test"], f"{name} / TEST int8")
        latency = bench_tflite(tflite_path)
    except Exception as e:
        print(f"int8 export/eval failed for {name}: {e!r}")

    n_params = sum(p.numel() for p in best.model.parameters())
    save_metrics(name, "13_yolo_nano", params=int(n_params),
                 int8_kb=round(os.path.getsize(tflite_path) / 1024, 1) if tflite_path else None,
                 val_map50=val["all"]["map50"], test_float=test_float, test_int8=test_int8,
                 latency_ms=latency, pt_path=pt_path, tflite_path=tflite_path)
    return best, val

In [ ]:
yolo11, yolo11_val = run_ultralytics("yolo11n.pt")

In [ ]:
yolo5, yolo5_val = run_ultralytics("yolov5nu.pt")

## Download the results (only when not saving to Drive)

With `USE_DRIVE = False`, the models and metrics live only on this session's disk and vanish
when it ends. This cell zips `models/parts_det/` and downloads it to your computer. With Drive
on, it does nothing, because the files are already in `MyDrive/Argus/models/parts_det/`.

**Shared verbatim by notebooks 12 / 13 / 14.**

In [ ]:
if IN_COLAB and not USE_DRIVE and not SMOKE:
    from google.colab import files
    archive = shutil.make_archive(f"{WORK_DIR}/parts_det_results_{VERSION_STR}", "zip", models_folder)
    files.download(archive)
else:
    print(f"results are in {models_folder}")